In [0]:
%pip install requests beautifulsoup4

In [0]:
import math
import re
from typing import Optional

import requests
from bs4 import BeautifulSoup

import sys, os
sys.path.append(os.path.abspath(".."))

from core.constants import BASE_URL, HEADERS
from core.schemas import RealEstateListing
from core.gather import Gather

In [0]:
def get_max_pages(category_url: str, page_size: int = 20) -> int:
    resp = requests.get(f"{category_url}0/", headers=HEADERS)
    soup = BeautifulSoup(resp.text, "html.parser")

    header_text = soup.select_one(".inzeratynadpis").get_text()
    match = re.search(r"z\s+([\d\s\xa0]+)", header_text)
    if not match:
        return 1

    total = int(match.group(1).replace(" ", "").replace("\xa0", ""))
    return math.ceil(total / page_size)


def _find_detail_row(soup: BeautifulSoup, label: str):
    detail_table = soup.select_one("td.listadvlevo")
    if not detail_table:
        return None
    for row in detail_table.find_all("tr"):
        cells = row.find_all("td")
        if cells and label in cells[0].get_text():
            return cells[-1]
    return None


def scrape_listing_detail(url: str, property_type: str, transaction_type: str, source: str = "bazos") -> RealEstateListing:
    resp = requests.get(url, headers=HEADERS)
    soup = BeautifulSoup(resp.text, "html.parser")

    listing_id = url.rstrip("/").split("/")[-2]
    title = soup.select_one("h1.nadpisdetail").get_text(strip=True)

    description_div = soup.select_one("div.popisdetail")
    description = description_div.get_text(separator="\n", strip=True) if description_div else ""

    price_cell = _find_detail_row(soup, "Cena")
    price = price_cell.get_text(strip=True) if price_cell else ""

    location_cell = _find_detail_row(soup, "Lokalita")
    location_zip, location_city = None, None
    if location_cell:
        links = location_cell.find_all("a")
        if len(links) >= 2:
            location_zip = links[0].get_text(strip=True)
            location_city = links[1].get_text(strip=True)

    header_span = soup.select_one(".inzeratydetnadpis .velikost10")
    posted_date = None
    if header_span:
        match = re.search(r"\[(.*?)\]", header_span.get_text())
        if match:
            posted_date = match.group(1)

    images = []
    for img in soup.select(".carousel-cell-image"):
        src = img.get("src") or img.get("data-flickity-lazyload")
        if src:
            images.append(src)

    return RealEstateListing(
        listing_id=listing_id,
        source=source,
        source_url=url,
        property_type=property_type,
        transaction_type=transaction_type,
        title=title,
        description=description,
        price=price,
        location_city=location_city,
        location_zip=location_zip,
        posted_date=posted_date,
        images=images,
    )


def collect_and_scrape(category_url: str, property_type: str, transaction_type: str, gather: Gather, max_pages: Optional[int] = None) -> None:
    total_pages = get_max_pages(category_url)
    pages_to_scrape = min(total_pages, max_pages) if max_pages else total_pages

    for page_num in range(pages_to_scrape):
        offset = page_num * 20
        page_url = f"{category_url}{offset}/"
        resp = requests.get(page_url, headers=HEADERS)
        soup = BeautifulSoup(resp.text, "html.parser")

        listings = soup.select(".inzeraty.inzeratyflex")
        if not listings:
            break

        page_urls = [
            BASE_URL + item.select_one(".nadpis a")["href"]
            for item in listings
            if item.select_one(".nadpis a") and item.select_one(".nadpis a").get("href")
        ]

        for url in page_urls:
            try:
                listing = scrape_listing_detail(url, property_type, transaction_type)
                gather.push(listing)
            except Exception as e:
                print(f"Exception on {url}: {e}")


def scrape_all_categories(base_urls: dict, gather: Gather, max_pages: Optional[int] = None) -> None:
    for transaction_type, categories in base_urls.items():
        for property_type, category_url in categories.items():
            print(f"Scraping {property_type}/{transaction_type}...")
            collect_and_scrape(category_url, property_type, transaction_type, gather, max_pages=max_pages)

In [0]:
import sys, os
sys.path.append(os.path.abspath(".."))

from core.constants import BASE_URLS

gather = Gather(source="bazos", is_debug=True)
scrape_all_categories(BASE_URLS, gather, max_pages=2)
gather.push_to_bronze()